# Live Target Tracker: Polaris Australis (Sigma Octantis)

This notebook is a **live-location tracker** for **Polaris Australis**.  
It does **not** generate images. It is focused on:

- setting your observing site
- computing the target's **current altitude and azimuth**
- showing whether it is **above your horizon right now**
- tracking its position live in a text/table view
- previewing the next observing window in a table

> Polaris Australis is the southern pole star and is also known as **Sigma Octantis**.


## What this notebook does

This notebook follows a clean, step-by-step workflow:

1. Install/import the astronomy tools  
2. Define your observing site  
3. Define the target (**Polaris Australis / Sigma Octantis**)  
4. Compute the **live sky position**  
5. Run a **refreshing live monitor**  
6. Preview the next few hours as a table  
7. Summarize whether the target is realistically observable from your site

**No sky images are produced.**


In [ ]:
# Run once in a fresh notebook environment
%pip install -q -U astropy pandas ipywidgets


In [ ]:
from dataclasses import dataclass
from zoneinfo import ZoneInfo
from datetime import datetime
import time

import numpy as np
import pandas as pd

import astropy.units as u
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
from astropy.time import Time

from IPython.display import display, clear_output

print("Imports loaded.")


## Observer configuration

Update this cell with **your observing location**.

The default values below use **Pune, India** as an example.  
If you are observing from somewhere else, replace the latitude, longitude, elevation, and timezone.


In [ ]:
@dataclass
class ObserverConfig:
    name: str = "Pune, India"
    latitude_deg: float = 18.5204
    longitude_deg: float = 73.8567
    elevation_m: float = 560.0
    timezone: str = "Asia/Kolkata"

observer = ObserverConfig()
observer


## Target definition

This notebook uses **Polaris Australis = Sigma Octantis** with fixed ICRS/J2000 coordinates.

You can keep this as-is for Polaris Australis.


In [ ]:
TARGET_NAME = "Polaris Australis (Sigma Octantis)"

# ICRS / J2000 coordinates
# RA  = 21h 08m 46.20s
# Dec = -88d 57m 23.0s
target = SkyCoord("21h08m46.20s", "-88d57m23.0s", frame="icrs")

print(TARGET_NAME)
print("RA :", target.ra.to_string(unit=u.hour, sep=":"))
print("Dec:", target.dec.to_string(unit=u.deg, sep=":"))


## Helper functions

These functions convert the target coordinates into your **live local sky position**.


In [ ]:
def make_location(cfg: ObserverConfig) -> EarthLocation:
    return EarthLocation(
        lat=cfg.latitude_deg * u.deg,
        lon=cfg.longitude_deg * u.deg,
        height=cfg.elevation_m * u.m,
    )

def visibility_label(alt_deg: float) -> str:
    if alt_deg <= 0:
        return "Below horizon"
    if alt_deg < 20:
        return "Visible, but low"
    if alt_deg < 40:
        return "Visible"
    return "Well placed"

def local_timestamp_str(ts: Time, timezone_name: str) -> str:
    dt_utc = ts.to_datetime(timezone=ZoneInfo("UTC"))
    dt_local = dt_utc.astimezone(ZoneInfo(timezone_name))
    return dt_local.strftime("%Y-%m-%d %H:%M:%S %Z")

def target_status(cfg: ObserverConfig, when: Time | None = None) -> dict:
    if when is None:
        when = Time.now()

    location = make_location(cfg)
    frame = AltAz(obstime=when, location=location)
    altaz = target.transform_to(frame)

    lst = when.sidereal_time("apparent", longitude=location.lon)
    hour_angle = (lst - target.ra).wrap_at(24 * u.hourangle)

    alt_deg = float(altaz.alt.deg)
    az_deg = float(altaz.az.deg)

    airmass = np.nan
    if alt_deg > 0:
        try:
            airmass = float(altaz.secz.value)
        except Exception:
            airmass = np.nan

    return {
        "Site": cfg.name,
        "Local time": local_timestamp_str(when, cfg.timezone),
        "UTC time": when.to_datetime(timezone=ZoneInfo("UTC")).strftime("%Y-%m-%d %H:%M:%S %Z"),
        "Target": TARGET_NAME,
        "Altitude (deg)": round(alt_deg, 3),
        "Azimuth (deg)": round(az_deg, 3),
        "Hour angle (hours)": round(hour_angle.to(u.hourangle).value, 3),
        "Airmass": None if np.isnan(airmass) else round(airmass, 3),
        "Status": visibility_label(alt_deg),
    }

def show_live_status(cfg: ObserverConfig):
    row = target_status(cfg)
    df = pd.DataFrame([row])
    display(df)

show_live_status(observer)


## Live monitor

This cell refreshes the target position every few seconds in a **table view only**.

- `refresh_seconds`: how often to update
- `iterations`: how many updates to show

You can stop it manually with **Interrupt / Stop** in your notebook UI.


In [ ]:
def live_monitor(cfg: ObserverConfig, refresh_seconds: int = 5, iterations: int = 24):
    try:
        for i in range(iterations):
            clear_output(wait=True)
            print(f"Tracking: {TARGET_NAME}")
            print(f"Observer: {cfg.name}")
            print(f"Update {i+1} / {iterations}   |   refresh every {refresh_seconds} s")
            print("-" * 72)
            show_live_status(cfg)
            time.sleep(refresh_seconds)
    except KeyboardInterrupt:
        print("Live monitor stopped.")

# Example:
# live_monitor(observer, refresh_seconds=5, iterations=24)


## Track the next few hours

This gives you a table of future positions so you can see whether the target rises, sets, or stays near the pole.


In [ ]:
def track_forward(cfg: ObserverConfig, hours: int = 12, step_minutes: int = 30) -> pd.DataFrame:
    base = Time.now()
    offsets = np.arange(0, hours * 60 + 1, step_minutes) * u.min
    rows = [target_status(cfg, base + offset) for offset in offsets]
    return pd.DataFrame(rows)

forward_df = track_forward(observer, hours=12, step_minutes=30)
display(forward_df)


## Observability summary

This cell gives a quick summary over the next 24 hours.


In [ ]:
def observability_summary(cfg: ObserverConfig, hours: int = 24, step_minutes: int = 10):
    df = track_forward(cfg, hours=hours, step_minutes=step_minutes)
    altitudes = df["Altitude (deg)"].astype(float)

    ever_visible = (altitudes > 0).any()
    always_visible = (altitudes > 0).all()

    summary = {
        "Site": cfg.name,
        "Min altitude over window (deg)": round(float(altitudes.min()), 3),
        "Max altitude over window (deg)": round(float(altitudes.max()), 3),
        "Ever above horizon?": "Yes" if ever_visible else "No",
        "Always above horizon?": "Yes" if always_visible else "No",
    }

    display(pd.DataFrame([summary]))

    if not ever_visible:
        print("\nResult: Polaris Australis does not rise above the horizon from this site in the selected window.")
    elif always_visible:
        print("\nResult: Polaris Australis is circumpolar from this site and remains above the horizon.")
    else:
        print("\nResult: Polaris Australis is observable for part of the selected window.")

observability_summary(observer)


## Optional: test a southern-hemisphere observing site

If you want to see the same target from a location where it is usually observable, try a southern site like Sydney.


In [ ]:
southern_site = ObserverConfig(
    name="Sydney, Australia",
    latitude_deg=-33.8688,
    longitude_deg=151.2093,
    elevation_m=58.0,
    timezone="Australia/Sydney",
)

print("Current status from Sydney:")
show_live_status(southern_site)

print("\n24-hour summary from Sydney:")
observability_summary(southern_site)


## Notes

- In the **northern hemisphere**, Polaris Australis is usually **not observable**, because it lies very close to the **south celestial pole**.
- In the **southern hemisphere**, it can be visible and often behaves like a near-polar target.
- For a different target later, you only need to replace the `target = SkyCoord(...)` line.

This notebook is intentionally **text/table based only**.
